In [1]:
# Kernel Warm UP
2+3

5

## Goal 1. Ethernet Header: Packet size, Destination MAC address, Source MAC address, Ethertype.


In [ ]:
from scapy.all import rdpcap, IP, TCP, UDP
import scapy
import pandas as pd
import argparse
from scapy.all import IP, IPv6, ARP


packets = rdpcap('test_wifi.pcap')


In [52]:
temp_dict = {"packet_size_bytes": [], "destination_mac_address": [], "source_mac_address": [], "ether_type": []}
for pkt in packets:
    packet_size = len(pkt)
    destination_mac_address = pkt["Ether"].dst if "Ether" in pkt else None
    source_mac_address = pkt["Ether"].src if "Ether" in pkt else None
    ether_type = scapy.layers.l2.ETHER_TYPES[pkt["Ether"].type] if "Ether" in pkt else None

    temp_dict["packet_size_bytes"].append(packet_size)
    temp_dict["destination_mac_address"].append(destination_mac_address)
    temp_dict["source_mac_address"].append(source_mac_address)
    temp_dict["ether_type"].append(ether_type)
pd.DataFrame(temp_dict)

,packet_size_bytes,destination_mac_address,source_mac_address,ether_type
0,80,26:80:df:ad:5f:7a,30:b6:4f:86:f6:ed,IPv4
1,86,ff:ff:ff:ff:ff:ff,26:80:df:ad:5f:7a,IPv4
2,77,26:80:df:ad:5f:7a,30:b6:4f:86:f6:ed,IPv4
3,120,10:70:fd:02:40:e0,26:80:df:ad:5f:7a,IPv4
4,723,10:70:fd:02:40:e0,26:80:df:ad:5f:7a,IPv4
...,...,...,...,...
28728,77,26:80:df:ad:5f:7a,30:b6:4f:86:f6:ed,IPv4
28729,134,26:80:df:ad:5f:7a,2c:21:31:4e:6b:78,IPv6
28730,54,10:70:fd:02:40:e0,26:80:df:ad:5f:7a,IPv4
28731,80,26:80:df:ad:5f:7a,30:b6:4f:86:f6:ed,IPv4


## Goal 2. IP Header: Version, Header length, Type of service, Total length, Identification, Flags, Fragment offset, Time to live, Protocol, Header checksum, Source and Destination IP addresses.

In [67]:
test_pkt = packets[0]
import datetime
import socket
readable_time = datetime.datetime.fromtimestamp(float(test_pkt.time)).strftime('%H:%M:%S.%f')
print(readable_time)

07:57:12.468186


In [ ]:
from scapy.all import IP, IPv6, ARP

print("\n IP Header Information:")
ip_temp_dict = {"version": [], "header_length_bytes": [], "Type of Service": [], "total_length_bytes": [], "identification": [], "flags": [], "fragment_offset": [], "time_to_live": [], "protocol": [], "header_checksum": [], "source_ip_address": [], "destination_ip_address": []}

for pkt in packets:
    if IP in pkt:
        ip_layer = pkt[IP]
        ip_temp_dict["version"].append(ip_layer.version)
        ip_temp_dict["header_length_bytes"].append(ip_layer.ihl * 4)
        ip_temp_dict["Type of Service"].append(ip_layer.tos)
        ip_temp_dict["total_length_bytes"].append(ip_layer.len)
        ip_temp_dict["identification"].append(ip_layer.id)
        ip_temp_dict["flags"].append(ip_layer.flags)
        ip_temp_dict["fragment_offset"].append(ip_layer.frag)
        ip_temp_dict["time_to_live"].append(ip_layer.ttl)
        ip_temp_dict["protocol"].append(IP(proto=ip_layer.proto).sprintf("%IP.proto%"))
        ip_temp_dict["header_checksum"].append(ip_layer.chksum)
        ip_temp_dict["source_ip_address"].append(ip_layer.src)
        ip_temp_dict["destination_ip_address"].append(ip_layer.dst)
    elif IPv6 in pkt:
        ipv6_layer = pkt[IPv6]
        ip_temp_dict["version"].append(ipv6_layer.version)
        ip_temp_dict["header_length_bytes"].append(40)  # IPv6 header is always fixed at 40 bytes
        ip_temp_dict["Type of Service"].append(ipv6_layer.tc)
        ip_temp_dict["total_length_bytes"].append(ipv6_layer.plen + 40)  # payload + fixed header
        ip_temp_dict["identification"].append(None)  # IPv6 does not have an identification field
        ip_temp_dict["flags"].append(None)  # IPv6 does not have flags
        ip_temp_dict["fragment_offset"].append(None)  # IPv6 does not have fragment offset
        ip_temp_dict["time_to_live"].append(ipv6_layer.hlim)
        ip_temp_dict["protocol"].append(IP(proto=ipv6_layer.nh).sprintf("%IP.proto%"))
        ip_temp_dict["header_checksum"].append(None)  # IPv6 does not have a header checksum
        ip_temp_dict["source_ip_address"].append(ipv6_layer.src)
        ip_temp_dict["destination_ip_address"].append(ipv6_layer.dst)
    elif ARP in pkt:
        arp_layer = pkt[ARP]
        ip_temp_dict["version"].append(None)              # ARP has no IP version
        ip_temp_dict["header_length_bytes"].append(None)  # ARP has no IP header
        ip_temp_dict["Type of Service"].append(None)
        ip_temp_dict["total_length_bytes"].append(None)
        ip_temp_dict["identification"].append(None)
        ip_temp_dict["flags"].append(None)
        ip_temp_dict["fragment_offset"].append(None)
        ip_temp_dict["time_to_live"].append(None)
        ip_temp_dict["protocol"].append("arp")
        ip_temp_dict["header_checksum"].append(None)
        ip_temp_dict["source_ip_address"].append(arp_layer.psrc)  # sender IP (from ARP payload)
        ip_temp_dict["destination_ip_address"].append(arp_layer.pdst)  # target IP (from ARP payload)

ip_df = pd.DataFrame(ip_temp_dict)

int_columns = ["version", "header_length_bytes", "Type of Service", "total_length_bytes",
               "identification", "fragment_offset", "time_to_live", "header_checksum"]

for col in int_columns:
    ip_df[col] = ip_df[col].astype("Int64")



 IP Header Information:
       version  header_length_bytes  Type of Service  total_length_bytes  \
0          4.0                 20.0              0.0                66.0   
1          4.0                 20.0              0.0                72.0   
2          4.0                 20.0              0.0                63.0   
3          4.0                 20.0              2.0               106.0   
4          4.0                 20.0              2.0               709.0   
...        ...                  ...              ...                 ...   
28728      4.0                 20.0              0.0                63.0   
28729      6.0                 40.0              0.0               120.0   
28730      4.0                 20.0              0.0                40.0   
28731      4.0                 20.0              0.0                66.0   
28732      4.0                 20.0              0.0                52.0   

       identification flags  fragment_offset  time_to_live   p

In [100]:
for pkt in packets[:5]:
    print(repr(pkt))


<Ether  dst=26:80:df:ad:5f:7a src=30:b6:4f:86:f6:ed type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=66 id=3881 flags=DF frag=0 ttl=62 proto=udp chksum=0xe4b0 src=129.21.215.193 dst=239.255.255.250 |<UDP  sport=52002 dport=15600 len=46 chksum=0x930 |<Raw  load=b'SEARCH BSDP/0.1\nDEVICE=0\nSERVICE=4097\n' |>>>>
<Ether  dst=ff:ff:ff:ff:ff:ff src=26:80:df:ad:5f:7a type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=72 id=65095 flags= frag=0 ttl=64 proto=udp chksum=0x4c77 src=10.118.26.251 dst=10.118.255.255 |<UDP  sport=57621 dport=57621 len=52 chksum=0x34e0 |<Raw  load=b'SpotUdp0\xbel\x04\xdc t\xebs\x00\x01\x00\x04H\x95\xc2\x03\xb0\xcb<c\xfb\xe4\x94N\xcd\xa1x~O\x99\xb7\x04\xb1\xdb\xfaN' |>>>>
<Ether  dst=26:80:df:ad:5f:7a src=30:b6:4f:86:f6:ed type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=63 id=1655 flags=DF frag=0 ttl=63 proto=udp chksum=0xf765 src=129.21.204.193 dst=239.255.255.250 |<UDP  sport=34232 dport=15600 len=43 chksum=0xb6e6 |<Raw  load=b'SEARCH BSDP/0.1\nDEVICE=0\nSERVICE=1\n' |>>>>
<Ether  

[<Flag 2 (DF)>,
 <Flag 0 ()>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 0 ()>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 0 ()>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 0 ()>,
 <Flag 2 (DF)>,
 <Flag 0 ()>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <Flag 2 (DF)>,
 <